In [1]:
import time
from nba_api.stats.endpoints import leaguedashplayerstats
from google.cloud import bigquery
import pandas as pd

# 過去シーズンデータの一括取得

In [2]:
PROJECT_ID = "invertible-vine-477701-j8"
PLAYER_SEASON_STATS_TABLE_ID = "invertible-vine-477701-j8.nba_stats.player_season_stats"

client = bigquery.Client(project=PROJECT_ID)

# 試しに接続テスト
datasets = list(client.list_datasets())
if datasets:
    print("接続成功！データセット一覧:")
    for ds in datasets:
        print(f"- {ds.dataset_id}")
else:
    print("接続成功（データセットはまだありません）")

/Users/tokuyoshi/private-nba-analytics/.venv/lib/python3.11/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


接続成功！データセット一覧:
- nba_stats


In [3]:
stats = leaguedashplayerstats.LeagueDashPlayerStats(
        season="2010-11",
        per_mode_detailed="PerGame" # 1試合平均で取得
    )
df = stats.get_data_frames()[0]

df

,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,...,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,TEAM_COUNT
0,201985,AJ Price,AJ,1610612754,IND,24.0,50,22,28,0.440,...,308,351,198,240,161,274,224,27,264,1
1,201166,Aaron Brooks,Aaron,1610612756,PHX,26.0,59,26,33,0.441,...,62,205,131,132,377,179,154,27,164,2
2,201189,Aaron Gray,Aaron,1610612740,NOH,26.0,41,21,20,0.512,...,270,121,348,362,145,326,154,27,332,1
3,201151,Acie Law,Acie,1610612744,GSW,26.0,51,20,31,0.392,...,214,357,265,321,368,335,224,27,340,2
4,1733,Al Harrington,Al,1610612743,DEN,31.0,73,45,28,0.616,...,103,55,199,137,97,176,181,27,156,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
447,201146,Yi Jianlian,i Jianlian,1610612764,WAS,23.0,63,17,46,0.270,...,254,230,278,259,409,264,129,27,269,1
448,202220,Zabian Dowdell,Zabian,1610612756,PHX,26.0,24,14,10,0.583,...,387,345,310,288,318,299,224,27,304,1
449,2216,Zach Randolph,Zach,1610612763,MEM,29.0,75,43,32,0.573,...,4,123,31,21,46,17,4,27,15,1
450,2585,Zaza Pachulia,Zaza,1610612737,ATL,27.0,79,43,36,0.544,...,119,124,140,306,304,281,111,27,288,1


In [4]:
def fetch_and_load(season):
    # 1. nba_apiからシーズン単位の集計データを取得
    # 例：全選手のレギュラーシーズンのスタッツ
    stats = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        per_mode_detailed='PerGame' # 1試合平均で取得
    )
    df = stats.get_data_frames()[0]
    
    # シーズン識別用の列を追加
    df['SEASON_ID'] = season
    
    # 2. BigQueryへロード
    job_config = bigquery.LoadJobConfig(write_disposition="WRITE_APPEND")
    job = client.load_table_from_dataframe(df, destination=PLAYER_SEASON_STATS_TABLE_ID, job_config=job_config)
    job.result() # 完了まで待機
    print(f"Loaded {season} stats to BigQuery.")

# 過去シーズン分を取得（レート制限を考慮してsleepを入れる）
seasons = ["2000-01", "2001-02", "2002-03", "2003-04", "2004-05", "2005-06", "2006-07", "2007-08", "2008-09", "2009-10",
            "2010-11", "2011-12", "2012-13", "2013-14", "2014-15", "2015-16", "2016-17", "2017-18", "2018-19", "2019-20",
            "2020-21", "2021-22", "2022-23", "2023-24", "2024-25"]

for s in seasons:
    fetch_and_load(s)
    time.sleep(2) # NBA APIへの負荷軽減

Loaded 2000-01 stats to BigQuery.
Loaded 2001-02 stats to BigQuery.
Loaded 2002-03 stats to BigQuery.
Loaded 2003-04 stats to BigQuery.
Loaded 2004-05 stats to BigQuery.
Loaded 2005-06 stats to BigQuery.
Loaded 2006-07 stats to BigQuery.
Loaded 2007-08 stats to BigQuery.
Loaded 2008-09 stats to BigQuery.
Loaded 2009-10 stats to BigQuery.
Loaded 2010-11 stats to BigQuery.
Loaded 2011-12 stats to BigQuery.
Loaded 2012-13 stats to BigQuery.
Loaded 2013-14 stats to BigQuery.
Loaded 2014-15 stats to BigQuery.
Loaded 2015-16 stats to BigQuery.
Loaded 2016-17 stats to BigQuery.
Loaded 2017-18 stats to BigQuery.
Loaded 2018-19 stats to BigQuery.
Loaded 2019-20 stats to BigQuery.
Loaded 2020-21 stats to BigQuery.
Loaded 2021-22 stats to BigQuery.
Loaded 2022-23 stats to BigQuery.
Loaded 2023-24 stats to BigQuery.
Loaded 2024-25 stats to BigQuery.
